<style>
body {
    background: linear-gradient(135deg, #f3e7ff 0%, #e3f0ff 100%);
    font-family: 'Segoe UI', Arial, sans-serif;
}
div.custom-section {
    border: 2px solid #c77dff;
    padding: 18px;
    border-radius: 12px;
    background-color: #f8f0ff;
    box-shadow: 0 2px 8px rgba(123,44,191,0.08);
}
h2 {
    color: #7b2cbf;
}
</style>

<div class="custom-section">
<h2>Limpeza e Tratamento dos Dados</h2>
</div>

In [86]:
# Leitura do arquivo CSV
import os


nome_pasta = input("Digite seu nome (pasta): ")
base_path = f"../database/{nome_pasta}/Takeout/YouTube e YouTube Music"
os.listdir(base_path)

['canais',
 'chats ao vivo',
 'histórico',
 'inscrições',
 'metadados do vídeo',
 'music (library and uploads)',
 'playlists']

In [ ]:
from pathlib import Path
import re
from bs4 import BeautifulSoup
import pandas as pd

def ler_e_exibir_csv(caminho_arquivo):
   
    return pd.read_csv(caminho_arquivo)

# Dicionário de meses PT → número
meses = {
    "jan": "01",
    "fev": "02",
    "mar": "03",
    "abr": "04",
    "mai": "05",
    "jun": "06",
    "jul": "07",
    "ago": "08",
    "set": "09",
    "out": "10",
    "nov": "11",
    "dez": "12"
}

def converter_data_pt(data):
    if pd.isna(data):
        return None
    
    # Exemplo: 30 de jan. de 2026, 07:35:17
    match = re.search(
        r"(\d{1,2}) de (\w+)\.? de (\d{4}), (\d{2}:\d{2}:\d{2})",
        data
    )
    
    if not match:
        return None
    
    dia, mes, ano, hora = match.groups()
    mes = meses.get(mes[:3].lower())
    
    if not mes:
        return None
    
    data_formatada = f"{ano}-{mes}-{dia.zfill(2)} {hora}"
    return pd.to_datetime(data_formatada)

def extrair_historico_youtube(caminho_arquivo):
    with open(caminho_arquivo, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    dados = []

    blocos = soup.find_all("div", class_="content-cell")

    for bloco in blocos:
        a_tag = bloco.find("a")
        
        if a_tag:
            titulo = a_tag.get_text(strip=True)
            link = a_tag["href"]
            
            texto_completo = bloco.get_text(separator="\n").split("\n")
            
            # Tipo de ação
            if "Watched" in texto_completo[0]:
                tipo = "watch"
            elif "Searched for" in texto_completo[0]:
                tipo = "search"
            else:
                continue
            
            # Extração da data
            data_linha = next((t for t in texto_completo if "BRT" in t), None)

            if data_linha:
                match = re.search(
                    r"\d{1,2} de \w+\.? de \d{4}, \d{2}:\d{2}:\d{2}",
                    data_linha
                )
                data = match.group(0) if match else None
            else:
                data = None
            
            dados.append([tipo, titulo, data, link])

    df = pd.DataFrame(dados, columns=["tipo", "titulo", "data", "link"])
    
    df["data"] = df["data"].apply(converter_data_pt)

    return df


In [88]:
pesquisa = extrair_historico_youtube(base_path+"/histórico/histórico de pesquisa.html")
print(pesquisa)

        tipo                                             titulo  \
0      watch                       Easy Because ClickUp (PT-BR)   
1      watch  Vem pro EAD da São Judas e aproveite condições...   
2      watch                   SAME26 CAP 28 VID Malandrão2 YT   
3     search             como estudar e aprender professor pier   
4     search                            como estudar e aprender   
...      ...                                                ...   
1629  search                                     biologia total   
1630  search                                       prof jubilut   
1631  search                                            jubilut   
1632  search                                        giz com giz   
1633  search                                     curso em video   

                    data                                               link  
0    2026-01-30 07:35:40        https://www.youtube.com/watch?v=cWrNNL6Fje4  
1    2026-01-30 07:01:14        https:/

In [89]:
visual = extrair_historico_youtube(base_path+"/histórico/histórico-de-visualização.html")
print(visual)

       tipo                                             titulo  \
0     watch               COMO ESTUDAR E APRENDER - Prof  Pier   
1     watch                       Easy Because ClickUp (PT-BR)   
2     watch                  Linha Seara Gourmet by Bridgerton   
3     watch  Vem pro EAD da São Judas e aproveite condições...   
4     watch  Aprendendo Inteligência - Prof Pierluigi Piazz...   
...     ...                                                ...   
3640  watch  Conceitos Fundamentais em Química - Brasil Escola   
3641  watch  RESUMÃO 1º Ano do ENSINO MÉDIO - Todas as Maté...   
3642  watch      melhores canais do youtube para estudar pt 2?   
3643  watch  hábitos que vão mudar sua vida (estudos, belez...   
3644  watch  como organizar a sua vida inteira neste início...   

                    data                                         link  
0    2026-01-30 07:35:46  https://www.youtube.com/watch?v=noXqEGIZak8  
1    2026-01-30 07:35:40  https://www.youtube.com/watch?v=cWrNN

In [90]:
inscricoes = ler_e_exibir_csv(base_path+"/inscrições/inscrições.csv")
print(inscricoes)

                 ID do canal  \
0   UC0BiVs5EYh57gzGVvhddjsA   
1   UC3YoQ-S3szZH265K913KGjw   
2   UC5Bn1EuTqaQNCCKo_9SAi_w   
3   UCCTHtVMXy9gQA8X_HOEJA2A   
4   UCFS6ofy6-uQWg6yHmGdlnwg   
5   UCG-eghyORxNrzSG8FkAWKyw   
6   UCLv7Gzc3VTO6ggFlXY0sOyw   
7   UC_eBg6o7KnGs23t6p3T9YAg   
8   UCd3ThZLzVDDnKSZMsbK0icg   
9   UCn9Erjy00mpnWeLnRqhsA1g   
10  UCnnlOUsq7ZiPMqIGGUrN7tA   
11  UCrWvhVmt0Qac3HgsjQK62FQ   

                                         URL do canal  \
0   http://www.youtube.com/channel/UC0BiVs5EYh57gz...   
1   http://www.youtube.com/channel/UC3YoQ-S3szZH26...   
2   http://www.youtube.com/channel/UC5Bn1EuTqaQNCC...   
3   http://www.youtube.com/channel/UCCTHtVMXy9gQA8...   
4   http://www.youtube.com/channel/UCFS6ofy6-uQWg6...   
5   http://www.youtube.com/channel/UCG-eghyORxNrzS...   
6   http://www.youtube.com/channel/UCLv7Gzc3VTO6gg...   
7   http://www.youtube.com/channel/UC_eBg6o7KnGs23...   
8   http://www.youtube.com/channel/UCd3ThZLzVDDnKS...   
9   http://ww

In [91]:
visual['data']=pd.to_datetime(visual['data'],dayfirst=True, errors='coerce')
pesquisa['data']=pd.to_datetime(pesquisa['data'],dayfirst=True, errors='coerce')

limite=pd.Timestamp.today()-pd.Timedelta(days=30)

visual=visual[visual['data']>=limite]
pesquisa=pesquisa[pesquisa['data']>=limite]

In [92]:
print(visual)

     tipo                                             titulo  \
0   watch               COMO ESTUDAR E APRENDER - Prof  Pier   
1   watch                       Easy Because ClickUp (PT-BR)   
2   watch                  Linha Seara Gourmet by Bridgerton   
3   watch  Vem pro EAD da São Judas e aproveite condições...   
4   watch  Aprendendo Inteligência - Prof Pierluigi Piazz...   
5   watch                   SAME26 CAP 28 VID Malandrão2 YT   
6   watch  🖥✏️ tutorial no Notion // como eu uso + criand...   
7   watch              Comprei meu material escolar de 2026!   
8   watch                         “The Gunther” gone wrong 😂   
9   watch  FAMOSOS QUE SÃO FEIOS, MAS TEM FILHOS LINDOS! ...   
10  watch               SCHOOL SUPPLIES SHOPPING 2025!! VLOG   
11  watch  ORGANIZANDO MEU MATERIAL ESCOLAR 2026 | minha ...   
12  watch            SAME26 CAP 1012 VID REGIAO SAO PAULO YT   
13  watch  Apache Spark (Data Analytics poderoso) // Dici...   

                  data                 

In [93]:
visual['titulo'].isna().sum()
pesquisa['titulo'].isna().sum()
inscricoes['Título do canal'].isna().sum()

# Nenhum dado nulo foi encontrado

np.int64(0)

In [94]:
visual=visual.drop_duplicates()
pesquisa=pesquisa.drop_duplicates()

In [95]:
import re

def limpar_texto(texto):
    texto=texto.lower()
    texto=re.sub(r'[^\w\s]','',texto)
    return texto

visual['titulo']=visual['titulo'].apply(limpar_texto)
pesquisa['titulo']=pesquisa['titulo'].apply(limpar_texto)

<style>
body {
    background: linear-gradient(135deg, #f3e7ff 0%, #e3f0ff 100%);
    font-family: 'Segoe UI', Arial, sans-serif;
}
div.custom-section {
    border: 2px solid #c77dff;
    padding: 18px;
    border-radius: 12px;
    background-color: #f8f0ff;
    box-shadow: 0 2px 8px rgba(123,44,191,0.08);
}
h2 {
    color: #7b2cbf;
}
</style>

<div class="custom-section">

<h2>Séries Temporais</h2>
</div>

In [96]:
visual=visual.sort_values('data')
visual['proximo_video']=visual['titulo'].shift(-1)

In [97]:
print(visual)

     tipo                                             titulo  \
13  watch  apache spark data analytics poderoso  dicionár...   
12  watch            same26 cap 1012 vid regiao sao paulo yt   
11  watch  organizando meu material escolar 2026  minha m...   
10  watch                 school supplies shopping 2025 vlog   
9   watch  famosos que são feios mas tem filhos lindos cu...   
8   watch                            the gunther gone wrong    
7   watch               comprei meu material escolar de 2026   
6   watch   tutorial no notion  como eu uso  criando uma ...   
5   watch                    same26 cap 28 vid malandrao2 yt   
4   watch  aprendendo inteligência  prof pierluigi piazzi...   
3   watch  vem pro ead da são judas e aproveite condições...   
2   watch                  linha seara gourmet by bridgerton   
1   watch                          easy because clickup ptbr   
0   watch                como estudar e aprender  prof  pier   

                  data                 